# NHLA 2-Side Classification

This notebook trains the dual-face board model. It follows the same workflow as the 1-side notebook:
1. tune hyperparameters on a short validation pass,
2. train the final board model with the best settings,
3. save the final metrics for model comparison.

Only run this after the 1-side training is validated.

In [ ]:
from pathlib import Path
import json
import sys
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_NAME = "26-8-18_75VAL"
CSV_PATH = PROJECT_ROOT / "data" / DATASET_NAME / "nhla_dataset" / "nhla_board_labels.csv"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = PROJECT_ROOT / "models" / f"NHLA_Classification_2Side_{RUN_TIMESTAMP}"
OPTUNA_DIR = PROJECT_ROOT / "logs" / "optuna"
TENSORBOARD_DIR = PROJECT_ROOT / "logs" / "tensorboard" / f"NHLA_Classification_2Side_{RUN_TIMESTAMP}"
METRICS_DIR = PROJECT_ROOT / "data" / "metrics" / f"NHLA_Classification_2Side_{RUN_TIMESTAMP}"

# ------------------------------------------------------------------
# Configuration - change these knobs as needed
# ------------------------------------------------------------------
# Input geometry: every face is resized by the same IMAGE_SCALE (native is 1 mm/px along the
# board, so 0.375 -> ~2.7 mm/px) and padded onto a CANVAS (height, width), so the network sees
# true board length and defect size. 512 x 1888 fits 99% of faces; larger ones are shrunk to fit.
IMAGE_SCALE = 0.375
CANVAS = (512, 1888)
BATCH_SIZE = 8            # boards per batch (= 16 face images)
NUM_WORKERS = 6
CUDA_DEVICE_INDEX = 0
NUM_EPOCHS = 10           # resnet18: ~95 min/epoch on all 551k boards at this canvas
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VAL_SPLIT = 0.2
SEED = 33
CHECKPOINT_INTERVAL = 2

# How the board model combines the two faces:
#   "shared"   one backbone per face (shared weights), features concatenated (recommended)
#   "stacked"  top and bottom stacked vertically into one image
#   "channels" legacy 6-channel input
TWO_SIDE_FUSION = "shared"
# Optional: path to a 1-side model_best.pt to initialise the shared backbone from.
# When set, the encoder is taken from that checkpoint and is not tuned by Optuna.
INIT_CHECKPOINT = None

# Shared decode-once image cache (built by the next cell, ~115 GB). Must live on a Linux
# filesystem, NOT on the Windows-mounted repo drive. None -> ~/.cache/nhla_images (or $NHLA_CACHE_DIR).
IMAGE_CACHE_DIR = None
CACHE_BUILD_WORKERS = 16

# Optuna tuning controls
OPTUNA_ENABLED = True
OPTUNA_TRIALS = 10
TUNE_EPOCHS = 4
TUNE_SUBSET_FRACTION = 0.05  # label-stratified fraction of boards per trial (~28k boards, ~5 min/epoch)
STUDY_NAME = f"nhla_2side_optuna_{RUN_TIMESTAMP}"
OPTUNA_DB_PATH = OPTUNA_DIR / "NHLA_Classification.db"
OPTUNA_SEARCH_SPACE = {
    "lr": {"low": 1e-5, "high": 1e-3, "log": True},
    "batch_size": [8, 16],
    "weight_decay": {"low": 1e-6, "high": 1e-2, "log": True},
    "encoder_name": ["resnet18", "resnet34"],  # resnet50 is ~4.4 h/epoch at this canvas
}

# Run naming
CHECKPOINT_NAME = f"NHLA_2Side_{RUN_TIMESTAMP}"
LOG_NAME = f"NHLA_2Side_{RUN_TIMESTAMP}"
METRICS_RUN_NAME = f"nhla_2side_{RUN_TIMESTAMP}"

# Model config
ENCODER_NAME = "resnet18"  # options: resnet18, resnet34, resnet50
PRETRAINED = True

print(f"CSV path: {CSV_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"TensorBoard dir: {TENSORBOARD_DIR}")
print(f"Metrics dir: {METRICS_DIR}")
print(f"Image scale: {IMAGE_SCALE}, canvas: {CANVAS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Num workers: {NUM_WORKERS}")
print(f"CUDA device index: {CUDA_DEVICE_INDEX}")
print(f"Fusion: {TWO_SIDE_FUSION}, init checkpoint: {INIT_CHECKPOINT}")
print(f"Encoder: {ENCODER_NAME}, pretrained: {PRETRAINED}")
print(f"Optuna enabled: {OPTUNA_ENABLED}, trials: {OPTUNA_TRIALS}, tune epochs: {TUNE_EPOCHS}, subset: {TUNE_SUBSET_FRACTION}")
print("Optuna search space:")
print(json.dumps(OPTUNA_SEARCH_SPACE, indent=2, default=str))

In [ ]:
# Build the shared decode-once image cache (one-time; resumable if interrupted).
# Covers every face referenced by the CSVs in nhla_dataset/, so the 1-side notebook reuses it.
# Re-running is a no-op once the cache is complete.
from notebooks.NHLA_Classification import build_image_cache_for_csv

cache_dir, cache_index = build_image_cache_for_csv(
    str(CSV_PATH),
    image_scale=IMAGE_SCALE,
    canvas=CANVAS,
    cache_root=IMAGE_CACHE_DIR,
    num_workers=CACHE_BUILD_WORKERS,
)
print(f"Cache: {cache_dir} ({len(cache_index):,} images)")

In [ ]:
import sys
import importlib
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import notebooks.NHLA_Classification as nhla_module
import notebooks.NHLA_Classification_2Side as nhla_2side_module
importlib.reload(nhla_module)
importlib.reload(nhla_2side_module)
train_2side = nhla_2side_module.train_2side

result = train_2side(
    csv_path=str(CSV_PATH),
    output_dir=str(OUTPUT_DIR),
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    val_split=VAL_SPLIT,
    seed=SEED,
    optuna_trials=OPTUNA_TRIALS if OPTUNA_ENABLED else 0,
    tune_epochs=TUNE_EPOCHS,
    encoder_name=ENCODER_NAME,
    pretrained=PRETRAINED,
    search_space=OPTUNA_SEARCH_SPACE,
    study_name=STUDY_NAME,
    db_path=str(OPTUNA_DB_PATH),
    checkpoint_name=CHECKPOINT_NAME,
    log_name=LOG_NAME,
    metrics_run_name=METRICS_RUN_NAME,
    tensorboard_dir=str(TENSORBOARD_DIR),
    metrics_dir=str(METRICS_DIR),
    checkpoint_interval=CHECKPOINT_INTERVAL,
    num_workers=NUM_WORKERS,
    cuda_device_index=CUDA_DEVICE_INDEX,
    tune_subset_fraction=TUNE_SUBSET_FRACTION,
    image_cache_dir=IMAGE_CACHE_DIR,
    image_scale=IMAGE_SCALE,
    canvas=CANVAS,
    two_side_fusion=TWO_SIDE_FUSION,
    init_checkpoint=INIT_CHECKPOINT,
)

print(json.dumps({
    "best_params": result["best_params"],
    "metrics": result["result"]["metrics"],
    "metrics_path": result["result"].get("metrics_path"),
}, indent=2))

## Notes

- Optuna is run before the final training step.
- The tuned parameters are saved to `best_params.json` in the metrics directory.
- Final training uses the best parameters rather than the initial defaults.
- The 2-side model keeps its board-level CSV and paired-face architecture.